In [1]:
import itertools, csv, time
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

### The "RGB" Revelation: 

The thermal images are black and white, but they were saved as 3-channel RGB files (copying the same gray data into Red, Green, and Blue).
Why this hurts: The AI was wasting 66% of its brainpower looking for colors that don't exist, and our training settings were accidentally adding fake color noise, confusing the model.

In [8]:
import os
from PIL import Image
from pathlib import Path

DATASET_ROOT = "./17_03_dataset"
splits = ["train", "valid", "test"] # Adjust if your folders are named 'val' instead of 'valid'

print("🔄 Starting conversion to true grayscale...")

total_converted = 0

for split in splits:
    img_dir = Path(DATASET_ROOT) / split / "images"
    
    if not img_dir.exists():
        print(f"⚠️ Skipped '{split}': Directory not found at {img_dir}")
        continue
        
    files = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
    
    if not files:
        print(f"⚠️ Skipped '{split}': No images found.")
        continue

    print(f"   Processing {split} ({len(files)} images)...")
    
    for img_path in files:
        try:
            img = Image.open(img_path)
            
            # Only convert if it's actually RGB
            if img.mode == 'RGB':
                gray_img = img.convert('L') # Convert to 8-bit grayscale (1 channel)
                gray_img.save(img_path)     # Overwrite original
                total_converted += 1
        except Exception as e:
            print(f"❌ Error converting {img_path}: {e}")

print(f"\n✅ Conversion Complete!")
print(f"   Converted {total_converted} images to true 1-channel grayscale.")
print(f"   Dataset is now optimized for thermal segmentation.")

🔄 Starting conversion to true grayscale...
   Processing train (411 images)...
   Processing valid (127 images)...
   Processing test (95 images)...

✅ Conversion Complete!
   Converted 633 images to true 1-channel grayscale.
   Dataset is now optimized for thermal segmentation.


In [9]:
# repartition of the images in the dataset
dataset_path = "./17_03_dataset"

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset_path, split, "images")
    print(split, "images:", len(os.listdir(img_dir)))

train images: 411
valid images: 127
test images: 95


In [10]:
from PIL import Image

# Path to one of your images
img_path = "./17_03_dataset/train/images/video_001_event1_A002_500_t-00000s_01_jpg.rf.8a958210d0c7c29fd721f77f48ca9876.jpg" 

img = Image.open(img_path)
width, height = img.size
channels = len(img.getbands()) # 1 = Grayscale, 3 = RGB

print(f"✅ Image Size: {width} x {height} pixels")
print(f"✅ Color Channels: {channels} ({'Grayscale' if channels == 1 else 'RGB'})")

✅ Image Size: 432 x 432 pixels
✅ Color Channels: 1 (Grayscale)


In [11]:
# download the model
model = YOLO("yolov8s-seg.pt")

YOLO automatically resized your images from 432 to 448 because 432 isn't divisible by 32. This is fine, but good to know for your report (you are technically training at 448px).

In [12]:
# train the model on our dataset
results = model.train(
    data="./17_03_dataset_grayscale/data.yaml",
    epochs=100,
    imgsz=432,
    batch=16,
    workers=4
)

New https://pypi.org/project/ultralytics/8.4.35 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./17_03_dataset_grayscale/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=432, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_sca

  8                  -1  1   1838080  ultralytics.nn.modules.block.C2f             [512, 512, 1, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  1    591360  ultralytics.nn.modules.block.C2f             [768, 256, 1]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  1    148224  ultralytics.nn.modules.block.C2f             [384, 128, 1]                 
 16                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128,

In [16]:


# ── 1. Configuration ──────────────────────────────────────────────────────────
# Paths to your trained models
MODEL_RGB_PATH = "./runs/segment/train/weights/best.pt"       # Original (Fake RGB)
MODEL_GRAY_PATH = "./runs/segment/train2/weights/best.pt"     # New (True Grayscale)

# Path to the dataset YAML (Ensure this points to the GRAYSCALE dataset if the gray model was trained on it)
# If the gray model was trained on the converted images, use the grayscale YAML.
# Both models can be evaluated on the same grayscale test set for fair comparison.
DATASET_YAML = "./17_03_dataset_grayscale/data.yaml" 

print(f"🚀 Loading models and evaluating on Test Set...")

# ── 2. Define Evaluation Function ─────────────────────────────────────────────
def evaluate_model(name, model_path):
    try:
        model = YOLO(model_path)
        
        # Run validation on the TEST set (unseen data)
        # plots=False speeds up the process since we just want numbers
        metrics = model.val(data=DATASET_YAML, split="test", verbose=False, plots=False)
        
        return {
            "Model": name,
            "mAP50-95 (Box)": f"{metrics.box.map:.4f}",
            "mAP50-95 (Mask)": f"{metrics.seg.map:.4f}", # Primary metric for segmentation
            "Recall (Mask)": f"{metrics.seg.r[0]:.4f}",   # How many whales found?
            "Precision (Mask)": f"{metrics.seg.p[0]:.4f}",# How many detections were correct?
            "False Negatives (Est)": int(297 * (1 - metrics.seg.r[0])) # Assuming 297 test instances
        }
    except Exception as e:
        print(f"❌ Error evaluating {name}: {e}")
        return None

# ── 3. Run Evaluation ─────────────────────────────────────────────────────────
results = []

# Evaluate RGB Model
print(f"   📊 Evaluating RGB Model (Fake 3-channel)...")
res_rgb = evaluate_model("RGB (Original)", MODEL_RGB_PATH)
if res_rgb: results.append(res_rgb)

# Evaluate Grayscale Model
print(f"   📊 Evaluating Grayscale Model (True 1-channel)...")
res_gray = evaluate_model("Grayscale (Optimized)", MODEL_GRAY_PATH)
if res_gray: results.append(res_gray)

# ── 4. Display Comparison Table ───────────────────────────────────────────────
if len(results) == 2:
    df_compare = pd.DataFrame(results)
    
    print("\n" + "="*80)
    print("🏆 HEAD-TO-HEAD COMPARISON (Test Set)")
    print("="*80)
    print(df_compare.to_string(index=False))
    print("="*80)
    
    # Simple logic to declare a winner
    mask_map_rgb = float(df_compare[df_compare['Model'].str.contains('RGB')]['mAP50-95 (Mask)'].values[0])
    mask_map_gray = float(df_compare[df_compare['Model'].str.contains('Gray')]['mAP50-95 (Mask)'].values[0])
    
    if mask_map_gray > mask_map_rgb:
        print(f"✅ WINNER: Grayscale Model! (Improvement: {mask_map_gray - mask_map_rgb:.4f} mAP)")
    else:
        print(f"⚠️ RESULT: RGB Model performed similarly or better. (Diff: {mask_map_gray - mask_map_rgb:.4f})")
        
else:
    print("❌ Could not complete comparison. Check file paths.")

🚀 Loading models and evaluating on Test Set...
   📊 Evaluating RGB Model (Fake 3-channel)...
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)


YOLOv8s-seg summary (fused): 86 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 927.3±251.0 MB/s, size: 11.1 KB)
val: Scanning /home/floreM/FlukePrint_YOLO/best_model/17_03_dataset_grayscale/test/labels.cache... 95 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 95/95 28.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 15.9it/s 0.4s.4s
                   all         95        297      0.813      0.732      0.805      0.594      0.824      0.734      0.806      0.501
Speed: 0.4ms preprocess, 0.9ms inference, 0.0ms loss, 0.7ms postprocess per image
   📊 Evaluating Grayscale Model (True 1-channel)...
Ultralytics 8.4.19 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (p